# 🌳 Árboles de decisión: un modelo que se puede leer (versión sencilla)
**Curso de IA · Bloque de Machine Learning · Sesión 8 (versión básica)**

---

## La idea: el juego de las 20 preguntas

¿Has jugado a "adivina quién"? Haces preguntas de **sí o no** ("¿lleva gafas?", "¿es mujer?") y, con cada respuesta, **descartas candidatos** hasta acertar.

Un **árbol de decisión** hace exactamente eso con los datos:

```
                 ¿Es mujer?
                /          \
              sí            no
             /                \
     ¿Va en 1ª o 2ª clase?    ¿Tiene menos de 9 años?
       /        \               /          \
  SOBREVIVE     ...         SOBREVIVE       MUERE
```

| Parte | Nombre | Qué es |
|---|---|---|
| El primer cuadro | **Raíz** | La primera pregunta, la más importante |
| Los cuadros del medio | **Nodos** | Preguntas siguientes |
| Los cuadros del final | **Hojas** | La respuesta (la predicción) |

> 🔑 **Es el modelo más fácil de explicar de todo el curso:** se puede **dibujar** y **leer como unas reglas** ("si... entonces..."). No necesita fórmulas.

## El problema de hoy: ¿quién sobrevive a un naufragio?

Datos de pasajeros de un barco ficticio (inspirado en el Titanic). Queremos **predecir si un pasajero sobrevivió** a partir de su sexo, su edad, su clase y si viajaba con familia.

## ¿Cómo decide qué pregunta hacer?

En cada paso **prueba muchas preguntas posibles** ("¿edad ≤ 9?", "¿edad ≤ 10?", "¿es mujer?"...) y se queda con la que **separa mejor** a los que sobreviven de los que no.

Para medir "separar bien" usa el **índice de Gini**, que mide **lo mezclado** que está un grupo:

| Grupo | Gini | Significa |
|---|---|---|
| 10 sobreviven, 0 mueren | **0** | **Puro**: no hay duda |
| 8 sobreviven, 2 mueren | 0,32 | Algo mezclado |
| 5 sobreviven, 5 mueren | **0,5** | **Máxima mezcla** |

Una buena pregunta **reduce el Gini**: crea grupos **más puros** que el grupo de partida.

## Los pasos de hoy

| Apartado | Paso | En una frase |
|---|------|--------------|
| 1-2 | Herramientas y un árbol diminuto | La primera pregunta con 12 pasajeros |
| 3 | Datos del naufragio | Cargar, mirar y dibujar |
| 4 | Texto a números | `sexo` → 0 / 1 |
| 5 | Separar entrenamiento y prueba | Guardar pasajeros "para el examen" |
| 6 | Entrenar el primer árbol | `fit` y `predict` |
| 7 | **Dibujar y leer el árbol** | Las reglas que ha aprendido |
| 8 | **Un árbol sin límites** | ¿Por qué memoriza? |
| 9 | Elegir la profundidad | Demasiado simple frente a demasiado grande |
| 10 | Medir y usar el modelo | Matriz de confusión y predicciones |
| 11 | Resumen | La receta |

## Cómo usar este cuaderno

Ejecuta cada celda con **Shift + Enter**, **en orden**. Si algo se estropea: *Entorno de ejecución → Reiniciar y ejecutar todo*. Antes de ejecutar, **lee y piensa qué va a salir**.

> 👩‍🏫 **Nota para el profesor**
>
> Versión **simplificada** del cuaderno de árboles de decisión. Se han quitado: el **cálculo del Gini a mano** (solo se explica), los **nulos y el texto sucio**, el `Pipeline`, la **poda (`ccp_alpha`)**, **`GridSearchCV`**, las **fronteras de decisión**, la **inestabilidad** y el **árbol de regresión**. Se centra en lo que más se aprovecha en clase: **dibujar el árbol, leerlo como reglas y ver el sobreajuste**.
>
> **Duración orientativa: unas 3 horas** con los ejercicios.

## 1. Preparar las herramientas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Todo listo ✅")

## 2. Un árbol diminuto: 12 pasajeros y una pregunta

Antes del caso completo, veamos **un árbol de una sola pregunta** con 12 pasajeros. Tenemos su **edad** y si **sobrevivieron** (1) o **murieron** (0):

In [ ]:
mini = pd.DataFrame({
    "edad":      [2, 5, 8, 15, 22, 28, 35, 41, 50, 58, 63, 70],
    "sobrevive": [1, 1, 1,  1,  0,  0,  1,  0,  0,  0,  0,  0],
})
mini.T

Entrenamos un árbol que **solo puede hacer una pregunta** (`max_depth=1`) y lo dibujamos:

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

arbol_mini = DecisionTreeClassifier(max_depth=1, random_state=0)
arbol_mini.fit(mini[["edad"]], mini["sobrevive"])

plt.figure(figsize=(8, 4))
plot_tree(arbol_mini, feature_names=["edad"], class_names=["Muere", "Sobrevive"], filled=True, fontsize=11)
plt.show()

### Cómo leer cada cuadro del dibujo

| Línea | Significado |
|---|---|
| `edad <= 18.5` | **La pregunta.** Si es verdad → rama **izquierda**. Si es falsa → rama **derecha** |
| `gini = ...` | Lo **mezclado** que está el grupo (0 = puro) |
| `samples = ...` | **Cuántos** pasajeros llegan a ese cuadro |
| `value = [a, b]` | Cuántos hay de cada clase: **[mueren, sobreviven]** |
| `class = ...` | La clase que **predice** ese cuadro (la mayoritaria) |

👀 **Mira el resultado:** el árbol ha encontrado la mejor pregunta: **"¿edad ≤ 18,5?"**. A la izquierda quedan **4 pasajeros, todos supervivientes (Gini = 0, puro)**. A la derecha, el resto, más mezclado. Es la pregunta que mejor separa los grupos.

Un árbol "de verdad" **repite esto en cada rama**, con **todas las variables a la vez**, hasta que ya no puede o no queremos que siga.

## 3. Cargar los datos del naufragio

Esta celda fabrica `naufragio.csv` con **800 pasajeros**.

| Columna | Significado |
|---|---|
| `clase` | Clase del billete: 1 (alta), 2 o 3 (baja) |
| `sexo` | `mujer` / `hombre` ← **texto** |
| `edad` | Edad |
| `familiares` | Familiares a bordo |
| `sobrevive` | `1` si sobrevivió, `0` si no ← **lo que queremos predecir** |

> 💡 No hace falta entender el código de esta celda.

In [ ]:
rng = np.random.default_rng(8)
n = 800

clase = rng.choice([1, 2, 3], n, p=[0.25, 0.20, 0.55])
sexo = rng.choice(["mujer", "hombre"], n, p=[0.35, 0.65])
edad = np.clip(rng.normal(30, 14, n), 1, 75).round(0)
familiares = rng.poisson(0.8, n)

mujer = (sexo == "mujer")
z = (-1.4
     + 2.7 * mujer
     + 1.3 * (clase == 1) + 0.4 * (clase == 2)
     + 1.8 * (edad < 10)
     - 0.025 * (edad - 30) * (~mujer)
     - 0.9 * (familiares >= 4))
sobrevive = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

pd.DataFrame({
    "clase": clase, "sexo": sexo, "edad": edad,
    "familiares": familiares, "sobrevive": sobrevive,
}).to_csv("naufragio.csv", index=False, encoding="utf-8")

print("✅ Fichero 'naufragio.csv' creado")

In [ ]:
df = pd.read_csv("naufragio.csv")
print("Filas y columnas:", df.shape)
print("% de supervivientes:", round(df["sobrevive"].mean() * 100, 1))
df.head(8)

### ¿Qué patrones hay?

Calculamos el **porcentaje de supervivientes** según el sexo, la clase y la edad (la media de unos y ceros es el porcentaje de unos):

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

(df.groupby("sexo")["sobrevive"].mean() * 100).plot(kind="bar", ax=ejes[0], color=["tomato", "steelblue"])
ejes[0].set_title("% que sobrevive según el sexo")

(df.groupby("clase")["sobrevive"].mean() * 100).plot(kind="bar", ax=ejes[1], color="seagreen")
ejes[1].set_title("% que sobrevive según la clase")

tramos = pd.cut(df["edad"], bins=[0, 10, 20, 30, 40, 50, 80])
(df.groupby(tramos, observed=True)["sobrevive"].mean() * 100).plot(kind="bar", ax=ejes[2], color="darkorange")
ejes[2].set_title("% que sobrevive según la edad")

for e in ejes:
    e.set_ylabel("%")
    e.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

👀 **Patrones claros:** las **mujeres** sobreviven mucho más que los hombres, la **primera clase** mucho más que la tercera y los **niños pequeños** tienen una supervivencia muy alta. Un árbol debería **descubrir justo esas reglas**.

## 4. Texto a números

Los modelos solo entienden números. Convertimos `sexo` en una columna `sexo_mujer`: **1 si es mujer, 0 si es hombre**.

In [ ]:
df["sexo_mujer"] = (df["sexo"] == "mujer").astype(int)
df = df.drop(columns="sexo")
df.head()

> 🌳 Una ventaja de los árboles: **no hace falta escalar** las variables. Solo hacen preguntas del tipo "¿variable ≤ umbral?", así que dividir una variable entre 1.000 no cambia nada.

## 5. Separar entrenamiento y prueba

Como siempre: una parte para **aprender** y otra para **el examen**. Con `stratify=y` nos aseguramos de que **la proporción de supervivientes sea igual** en las dos.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["clase", "edad", "familiares", "sexo_mujer"]]
y = df["sobrevive"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Para entrenar:", len(X_train), "| Para el examen:", len(X_test))

## 6. Entrenar el primer árbol

Limitamos el árbol a **3 niveles de preguntas** (`max_depth=3`) para que se pueda leer.

In [ ]:
from sklearn.metrics import accuracy_score

arbol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol.fit(X_train, y_train)

predicciones = arbol.predict(X_test)

print(f"Accuracy en el examen: {accuracy_score(y_test, predicciones):.2f}")
print(f"(Un modelo que dijera siempre 'muere' acertaría: {(1 - y_test.mean()):.2f})")

👀 El árbol **mejora claramente** a un modelo que dijera siempre lo mismo. Y lo mejor: **vamos a poder ver por qué**.

## 7. Dibujar y leer el árbol

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(arbol, feature_names=list(X.columns), class_names=["Muere", "Sobrevive"],
          filled=True, rounded=True, fontsize=10)
plt.show()

### Cómo leer el dibujo

- **Cada caja** tiene una **pregunta** arriba. Si es **verdad** → rama **izquierda**; si es **falsa** → rama **derecha**.
- El **color** indica la clase que predice (naranja = muere, azul = sobrevive) y su **intensidad**, lo **seguro** que está.
- Las cajas del final (**hojas**) no tienen pregunta: dan la **respuesta**.

### Pasarlo a reglas de texto

`export_text` convierte el árbol en reglas del tipo "si... entonces...":

In [ ]:
from sklearn.tree import export_text

print(export_text(arbol, feature_names=list(X.columns), class_names=["Muere", "Sobrevive"]))

👀 **Cada línea que acaba en `class:` es una hoja**, y el camino que lleva hasta ella es **una regla completa**. Por ejemplo, algo como:

> *"SI no es mujer **Y** su edad es mayor de unos 9 años **Y** ... → MUERE"*

> 🔎 **Detalle curioso:** verás que algunas preguntas llevan a **dos hojas con la misma respuesta** (por ejemplo, la pregunta por la clase en los hombres mayores de 9 años). El árbol las hace porque **separan grupos más puros** (una rama tiene más supervivientes que la otra), aunque la **predicción final** no cambie.

Esto es lo que hace **único** al árbol: **cualquier persona** puede entender **por qué** se predice lo que se predice, sin saber estadística.

## 8. Un árbol sin límites: el sobreajuste

¿Qué pasa si dejamos al árbol **hacer todas las preguntas que quiera**? Por defecto, `DecisionTreeClassifier` **sigue dividiendo hasta que todos los grupos son puros**:

In [ ]:
arbol_libre = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

print("Preguntas seguidas (profundidad):", arbol_libre.get_depth())
print("Número de hojas                 :", arbol_libre.get_n_leaves())
print()
print(f"Accuracy en entrenamiento: {accuracy_score(y_train, arbol_libre.predict(X_train)):.2f}")
print(f"Accuracy en el examen    : {accuracy_score(y_test, arbol_libre.predict(X_test)):.2f}")

🚨 **Fíjate:**
- En **entrenamiento acierta muchísimo más** (≈ 90 %): ha creado **cientos de hojas** (una por cada 3 pasajeros, más o menos), memorizando los datos. (No llega al 100 % solo porque hay pasajeros **idénticos** con resultados **distintos**, que ningún árbol puede separar.)
- **En el examen acierta bastante menos** (y menos que el árbol pequeño).

> 🧠 **Un árbol sin límites casi siempre llega a un acierto altísimo en entrenamiento**: eso **no significa que sea bueno**. Ha **memorizado**, no **aprendido**. Es el **sobreajuste**.

Para dibujarlo habría que imprimir un árbol con cientos de cajas, ilegible. Por eso **limitamos** la profundidad.

## 9. Elegir la profundidad

Probamos árboles de **profundidad 1 a 10** y medimos la accuracy en **entrenamiento** y en **el examen**:

In [ ]:
filas = []
for profundidad in range(1, 11):
    a = DecisionTreeClassifier(max_depth=profundidad, random_state=42).fit(X_train, y_train)
    filas.append({
        "profundidad": profundidad,
        "hojas": a.get_n_leaves(),
        "accuracy entrenamiento": round(accuracy_score(y_train, a.predict(X_train)), 3),
        "accuracy examen": round(accuracy_score(y_test, a.predict(X_test)), 3),
    })

tabla = pd.DataFrame(filas).set_index("profundidad")
tabla

In [ ]:
plt.plot(tabla.index, tabla["accuracy entrenamiento"], marker="o", color="seagreen", label="Entrenamiento")
plt.plot(tabla.index, tabla["accuracy examen"], marker="o", color="darkorange", label="Examen")
plt.xlabel("Profundidad máxima del árbol")
plt.ylabel("Accuracy")
plt.title("Accuracy según la profundidad")
plt.legend()
plt.show()

👀 **Cómo leerlo:**

| Profundidad | Qué pasa |
|---|---|
| **1-2** | Pocas preguntas: el modelo es **simple**. Ya acierta bastante (el sexo explica casi todo) |
| **3-5** | El examen se queda **más o menos igual** mientras el entrenamiento **sube**: las preguntas extra ya solo ajustan detalles |
| **Más de 6** | El entrenamiento sigue subiendo pero **el examen baja**: **sobreajuste** |

> 📏 Con pocos pasajeros de examen, **diferencias de 1-2 puntos son ruido**. No hay que obsesionarse con decimales: lo importante es la **forma** de la gráfica.

> 🎯 **Regla de oro:** entre árboles que rinden parecido, **elige el más pequeño**: se entiende mejor y generaliza mejor.

## 10. Medir y usar el modelo

### La matriz de confusión del árbol de profundidad 3

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, predicciones, display_labels=["Muere", "Sobrevive"], cmap="Blues", colorbar=False
)
plt.title("Matriz de confusión (árbol de profundidad 3)")
plt.show()

Recuerda: la diagonal (arriba-izquierda y abajo-derecha) son los **aciertos**; el resto son los **errores**.

### ¿Qué variables ha usado más?

`feature_importances_` indica **cuánto ha contribuido cada variable** a separar los grupos:

In [ ]:
importancias = pd.Series(arbol.feature_importances_, index=X.columns).sort_values()

importancias.plot(kind="barh", color="steelblue", figsize=(6, 3))
plt.title("Importancia de cada variable en el árbol")
plt.show()

importancias.sort_values(ascending=False).round(2)

👀 **`sexo_mujer`** es, con diferencia, la más importante (es la **raíz** del árbol). Le siguen la **edad** y la **clase**. Las variables que **no aparecen en ninguna pregunta** tienen importancia 0.

### Predecir un pasajero nuevo

`predict_proba` da la **probabilidad** de cada resultado:

In [ ]:
pasajeros = pd.DataFrame({
    "clase":      [1, 3, 3],
    "edad":       [30, 7, 40],
    "familiares": [0, 2, 0],
    "sexo_mujer": [1, 0, 0],
})

prob = arbol.predict_proba(pasajeros)[:, 1]

resultado = pasajeros.copy()
resultado["sexo"] = np.where(resultado["sexo_mujer"] == 1, "mujer", "hombre")
resultado["prob. de sobrevivir"] = (prob * 100).round(0).astype(int).astype(str) + " %"
resultado["predicción"] = np.where(prob >= 0.5, "Sobrevive", "Muere")
resultado[["sexo", "edad", "clase", "prob. de sobrevivir", "predicción"]]

Prueba a **cambiar la edad, la clase o el sexo** de un pasajero y fíjate en **qué rama del árbol** cambia el resultado: ¡se puede seguir **con el dedo** sobre el dibujo!

## 11. Resumen

### La receta del árbol de decisión

| Paso | Código |
|---|---|
| 1. Texto a números | `(df["sexo"] == "mujer").astype(int)` |
| 2. Separar | `train_test_split(X, y, test_size=0.2, stratify=y)` |
| 3. Entrenar (limitando la profundidad) | `DecisionTreeClassifier(max_depth=3).fit(X_train, y_train)` |
| 4. Dibujar | `plot_tree(arbol, feature_names=..., class_names=...)` |
| 5. Reglas en texto | `export_text(arbol, feature_names=...)` |
| 6. Predecir | `.predict(...)` / `.predict_proba(...)` |
| 7. Importancia de variables | `arbol.feature_importances_` |

### Ideas clave

1. Un árbol es una **secuencia de preguntas de sí/no** que acaba en una predicción.
2. En cada paso elige **la pregunta que mejor separa** los grupos (que reduce el Gini).
3. **Sin límites memoriza** los datos (acierto altísimo en entrenamiento, peor en el examen): hay que **frenarlo** (`max_depth`).
4. **No necesita escalar** las variables.
5. Su gran virtud: **se puede leer y explicar** como unas reglas.
6. Su gran defecto: **sobreajusta fácilmente** y un árbol solo es **inestable** (con otros datos, saldría otro árbol).

### 📖 Mini glosario

| Palabra | Significado sencillo |
|---|---|
| **Raíz** | La primera pregunta del árbol |
| **Hoja** | Un cuadro final, que da la respuesta |
| **Profundidad** | Cuántas preguntas seguidas como máximo |
| **Gini** | Lo mezclado que está un grupo (0 = puro) |
| **Sobreajuste** | Memorizar los datos en vez de aprender el patrón |

## ✍️ Ejercicios

**Nivel 1 – Para empezar**
1. Sigue con el dedo, en el dibujo del árbol de profundidad 3, el camino de **una mujer de primera clase** y escribe la regla completa. ¿Cuántos pasajeros de entrenamiento llegan a esa hoja?
2. Calcula a mano el Gini de un grupo con **6 supervivientes y 2 muertos** (`1 - p² - (1-p)²`, con p = 6/8).
3. ¿Qué profundidad te parece la más razonable según la tabla? ¿Por qué?

**Nivel 2 – Practicar**
4. Entrena un árbol con `max_depth=2` y dibújalo. ¿Qué dos preguntas hace? ¿Sigue acertando casi igual que el de profundidad 3?
5. Entrena un árbol con `min_samples_leaf=20`: obliga a que **cada hoja tenga al menos 20 pasajeros**. ¿Cuántas hojas tiene y qué accuracy saca?
6. Quita la columna `sexo_mujer` y vuelve a entrenar. ¿Cuál es la nueva raíz del árbol? ¿Empeora mucho?

**Nivel 3 – Reto**
7. Cambia `random_state=42` por otros valores en `train_test_split` y entrena el árbol de profundidad 3 con cada uno. ¿Cambia siempre la raíz? ¿Cambian las preguntas de debajo? (Estás viendo la **inestabilidad** de los árboles.)

---
### 🎯 ¿Y ahora qué?
Un árbol solo es **inestable**. La solución es **entrenar muchos árboles y que voten**: es el **Random Forest** (bosque aleatorio), la siguiente sesión.